# Benchmarking results (dissertation Ch. 3) -- charts

Charts for the benchmarking study (gp_reward-priors `HANDOFF_HP_SELECTION.md` §14).
DESCRIPTIVE: every number comes from the same collectors as the readouts
(`results/control_readout.py`, `eval_readout.py`, `norm_inventory.py`), so a cell appears
only when it is complete and audit-clean, and the **BNN/CVaR hold** (§14.1) applies: BNN
CVaR, BNN mean and ensemble CVaR are never fetched.

Run from `results/`. The first run fetches from W&B (several minutes) and caches to
`results/.cache/benchmark_data.pkl` (gitignored); set `REFRESH = True` to refetch after
new runs finish. Commit with all outputs cleared.

| chart | question |
|---|---|
| 1 | What do the terminals carry? Each reward with vs without terminals, each at its own selected index. |
| 2 | Do learned rewards beat a reward for moving fast? Each reward − the motion reward, with terminals. |
| 3 | Normalization: every seed-0 grid, with and without terminals, winners boxed. |
| 4 | Does the reported statistic matter? Max vs last-10 for every cell. |
| 5 | Learning curves with vs without terminals. |
| 6 | The ties ablation on the reward models. |
| 7 | PT on the fixed labeller vs the time-only reward (§14.10). |

In [ ]:
import json
import os
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np

HERE = os.path.abspath(".")
ROOT = os.path.dirname(HERE)
sys.path.insert(0, HERE)
sys.path.insert(0, ROOT)
import hold                      # noqa: E402  the BNN/CVaR hold (§14.1): INCLUDE stays False
import iql_score                 # noqa: E402
import eval_readout as R         # noqa: E402
import control_readout as CR     # noqa: E402
import norm_inventory as NI      # noqa: E402

_argv, sys.argv = sys.argv, [sys.argv[0]]
import phase2_sweeps as P        # noqa: E402  (parses argv at import)
sys.argv = _argv

REFRESH = False                  # True: refetch everything from W&B
CACHE = os.path.join(HERE, ".cache", "benchmark_data.pkl")
VARIANTS = R.VARIANTS            # ("medium-play", "medium-diverse", "large-play", "large-diverse")
SEEDS = R.SEEDS
FAMILIES = ("task_reward", "MR", "PT", "NMR", "ENS_MEAN", "motion")
LABEL = {"task_reward": "task reward", "MR": "MR", "PT": "PT", "NMR": "NMR", "ENS_MEAN": "ens. mean",
         "motion": "motion (path length)", "PT_TIMEONLY": "PT time-only (bug)"}
_cyc = plt.rcParams["axes.prop_cycle"].by_key()["color"]
COLOR = {f: _cyc[i % len(_cyc)] for i, f in enumerate(FAMILIES + ("PT_TIMEONLY",))}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
def retry(fn, *args, tries=3):
    # W&B reads time out now and then; a transient timeout should not lose the whole fetch
    import time
    for k in range(tries):
        try:
            return fn(*args)
        except Exception as e:                       # noqa: BLE001
            if k == tries - 1:
                raise
            print(f"  {fn.__name__}: {type(e).__name__}, retrying in 30 s")
            time.sleep(30)


def fetch():
    import wandb
    api = wandb.Api(timeout=120)
    winners = json.load(open(os.path.join(ROOT, "phase2_winners.json")))
    gpath = os.path.join(ROOT, P.GRID_WINNERS)
    gwin = json.load(open(gpath)) if os.path.exists(gpath) else {}
    anchor, _ = retry(R.collect, api, P, winners)
    timeonly, _ = retry(R.collect_timeonly, api, P)
    ctrl, _ = retry(CR.collect_controls, api, P, winners, gwin)
    grids = retry(NI.collect, api, P)
    return {"winners": winners, "gwin": gwin, "anchor": anchor, "timeonly": timeonly,
            "ctrl": ctrl, "grids": grids, "curves": {}}


if REFRESH or not os.path.exists(CACHE):
    D = fetch()
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    pickle.dump(D, open(CACHE, "wb"))
else:
    D = pickle.load(open(CACHE, "rb"))
print("anchor cells", len(D["anchor"]), "| control cells", len(D["ctrl"]), "| grids", len(D["grids"]))

In [ ]:
def ready(c):
    return c is not None and R.cell_ready(c) is None and sorted(c.get("max", {})) == SEEDS


def with_term(v, f):
    # the with-terminals cell: the anchor evaluation (motion: its control at index 3)
    if f == "motion":
        return D["ctrl"].get((v, "heuristic"))
    return D["anchor"].get((f"antmaze-{v}-v2", f))


def without_term(v, f):
    # the without-terminals cell at the reward's OWN no-terminals winner
    ctrl = D["ctrl"]
    if f in ("task_reward", "MR"):
        found = {a: w for a, w in D["gwin"].items()}
        return CR.at_winner(ctrl, found, D["winners"], v, f)
    if f == "motion":
        return ctrl.get((v, "atwin_heuristic"))
    return ctrl.get((v, f"atwin_{f}"))


def mean_sd(c, key="scores"):
    x = np.array([c[key][s] for s in SEEDS])
    return x.mean(), x.std(ddof=1)

## 1. What do the terminals carry?

Each reward with the dataset's terminals (filled) and without them (hollow), each at its own
selected normalization index; mean ± sd over seeds 1–10 (last-10 statistic). A missing marker
means that cell is not complete yet.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), sharey=True)
for ax, v in zip(axes, VARIANTS):
    for i, f in enumerate(FAMILIES):
        for j, (cell, filled) in enumerate(((with_term(v, f), True), (without_term(v, f), False))):
            if not ready(cell):
                continue
            m, s = mean_sd(cell)
            x = i + (-0.15 if filled else 0.15)
            ax.errorbar(x, m, yerr=s, fmt="o", color=COLOR[f], mfc=COLOR[f] if filled else "white",
                        mew=1.5, capsize=3)
        a, b = with_term(v, f), without_term(v, f)
        if ready(a) and ready(b):
            ax.plot([i - 0.15, i + 0.15], [mean_sd(a)[0], mean_sd(b)[0]], color=COLOR[f], lw=1, alpha=0.6)
    ax.set_xticks(range(len(FAMILIES)))
    ax.set_xticklabels([LABEL[f] for f in FAMILIES], rotation=40, ha="right")
    ax.set_title(v)
axes[0].set_ylabel("success (last-10 mean)")
fig.legend(handles=[plt.Line2D([], [], marker="o", ls="", color="k", label="with terminals"),
                    plt.Line2D([], [], marker="o", ls="", color="k", mfc="white", label="without terminals")],
           loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("With vs without terminals, each reward at its own selected index")
fig.tight_layout(rect=[0, 0.06, 1, 0.95])

## 2. Learned rewards against a reward for moving fast (with terminals)

Each reward − the motion reward (per-step torso path length, index 3), paired by seed;
95% exact bootstrap interval over seeds. Left of zero: the reward does worse than moving fast.

In [ ]:
fams = ("task_reward", "MR", "PT", "NMR", "ENS_MEAN", "PT_TIMEONLY")
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharex=True)
for ax, v in zip(axes, VARIANTS):
    h = D["ctrl"].get((v, "heuristic"))
    ys, labels = [], []
    for k, f in enumerate(fams):
        a = D["timeonly"].get((f"antmaze-{v}-v2", f)) if f == "PT_TIMEONLY" else with_term(v, f)
        if not (ready(a) and ready(h)):
            continue
        c = R.compare(a, h)
        y = len(ys)
        ax.plot([c["lower"], c["upper"]], [y, y], color=COLOR[f], lw=3)
        ax.plot(c["mean"], y, "o", color=COLOR[f])
        ys.append(y); labels.append(LABEL[f])
    ax.axvline(0, color="k", lw=1)
    ax.set_yticks(ys); ax.set_yticklabels(labels)
    ax.set_title(v + ("" if ready(h) else " (motion cell not complete)"))
    ax.set_xlabel("reward − motion (success)")
fig.suptitle("Each reward minus the motion reward, with terminals (95% exact bootstrap)")
fig.tight_layout()

## 3. Normalization grids, with and without terminals

Seed-0 last-10 score per `normalize_reward` index; the stage-4 winner is boxed. Rows are
variants; a blank row is a grid that is not complete (or not run).

In [ ]:
gfams = ("task_reward", "MR", "PT", "NMR", "ENS_MEAN", "heuristic")
fig, axes = plt.subplots(len(gfams), 2, figsize=(14, 2.7 * len(gfams)), constrained_layout=True)
for r, f in enumerate(gfams):
    for col, proto in enumerate(("with", "without")):
        ax = axes[r, col]
        n = 4 if f == "task_reward" else 8
        M = np.full((4, n), np.nan)
        wins = {}
        for i, v in enumerate(NI.VARIANTS):
            g = D["grids"].get((f, v, proto))
            if g is None:
                continue
            status, win, by = NI.grid_status(g["runs"], g["grid"])
            if status != "done":
                continue
            for idx, sc in by.items():
                M[i, idx] = sc
            wins[i] = win
        im = ax.imshow(M, vmin=0, vmax=0.8, cmap="viridis", aspect="auto")
        for i in range(4):
            for j in range(n):
                if not np.isnan(M[i, j]):
                    ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                            color="w" if M[i, j] < 0.45 else "k")
            if i in wins:
                ax.add_patch(plt.Rectangle((wins[i] - 0.5, i - 0.5), 1, 1, fill=False, ec="red", lw=2))
        ax.set_yticks(range(4)); ax.set_yticklabels(NI.VARIANTS, fontsize=8)
        ax.set_xticks(range(n)); ax.grid(False)
        ax.set_xlabel("normalize_reward index", fontsize=8)
        ax.set_title(f"{LABEL.get(f, f)} -- {'with' if proto == 'with' else 'without'} terminals", fontsize=10)
        if f == "heuristic" and proto == "with":
            ax.text(n / 2 - 0.5, 1.5, "no grid: index 3 fixed in advance (§13.14)", ha="center", va="center",
                    fontsize=9, color="gray")
fig.colorbar(im, ax=axes, shrink=0.4, label="seed-0 last-10 success")

## 4. Max vs last-10 for every cell

Each point is one cell (a reward on a variant, with or without terminals): its mean over
seeds of the run MAXIMUM (the PT paper's statistic) against the mean of the LAST TEN
evaluations (this study's). Distance above the diagonal is how much the max flatters a cell.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
pts = []
for (ds, f), c in list(D["anchor"].items()) + list(D["timeonly"].items()):
    if ready(c) and f in COLOR:
        pts.append((mean_sd(c)[0], mean_sd(c, "max")[0], f, "o"))
for (v, kind), c in D["ctrl"].items():
    if not ready(c):
        continue
    f = kind.split("_", 1)[1] if "_" in kind and kind.split("_", 1)[0] in ("noterm", "rerun", "atwin", "tiesnt") else kind
    f = {"heuristic": "motion"}.get(f, f)
    pts.append((mean_sd(c)[0], mean_sd(c, "max")[0], f if f in COLOR else "motion", "^"))
for x, y, f, mk in pts:
    ax.plot(x, y, mk, color=COLOR.get(f, "gray"), alpha=0.8)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim(0, 0.9); ax.set_ylim(0, 0.9)
ax.set_xlabel("last-10 mean"); ax.set_ylabel("max over the run")
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=COLOR[f], label=LABEL[f]) for f in COLOR]
          + [plt.Line2D([], [], marker="^", ls="", color="k", label="control cell (triangle)")], fontsize=8)
ax.set_title(f"Max vs last-10, {len(pts)} cells")

## 5. Learning curves with vs without terminals

Mean ± sd over seeds 1–10 of the evaluation success curve, for each reward family on each
variant, with terminals (solid) and without them at its own winner (dashed). Curves are
fetched once per sweep and cached.

In [ ]:
CURVE_FAMILIES = ("MR", "task_reward", "ENS_MEAN", "PT", "NMR", "motion")


def curves(sweep_id):
    if sweep_id not in D["curves"]:
        import wandb
        api = wandb.Api(timeout=120)
        def get():
            out = []
            for r in api.sweep(f"{P.ENTITY}/{P.IQL_PROJECT}/{sweep_id}").runs:
                if r.state == "finished":
                    steps, sc = iql_score.run_curve(r)
                    if len(sc) == iql_score.EVAL_POINTS:
                        out.append((np.asarray(steps), np.asarray(sc)))
            return out
        D["curves"][sweep_id] = retry(get)
        pickle.dump(D, open(CACHE, "wb"))
    return D["curves"][sweep_id]


fig, axes = plt.subplots(len(CURVE_FAMILIES), 4, figsize=(16, 2.6 * len(CURVE_FAMILIES)), sharex=True, sharey="row")
for r, f in enumerate(CURVE_FAMILIES):
    for ax, v in zip(axes[r], VARIANTS):
        for cell, ls in ((with_term(v, f), "-"), (without_term(v, f), "--")):
            if not ready(cell):
                continue
            cs = curves(cell["sweep"])
            if not cs:
                continue
            steps = cs[0][0]
            Y = np.stack([c[1] for c in cs])
            ax.plot(steps, Y.mean(0), ls, color=COLOR[f])
            ax.fill_between(steps, Y.mean(0) - Y.std(0), Y.mean(0) + Y.std(0), color=COLOR[f], alpha=0.12)
        if r == 0:
            ax.set_title(v)
        if v == VARIANTS[0]:
            ax.set_ylabel(LABEL[f])
for ax in axes[-1]:
    ax.set_xlabel("training step")
fig.suptitle("Success during training: with terminals (solid), without at its own winner (dashed)")
fig.tight_layout(rect=[0, 0, 1, 0.98])

## 6. The ties ablation on the reward models (§14.29)

From `gp_reward-priors/calibration_results/ties/ties_measurements.json` (measured on the box,
copied over). Left: accuracy on the tie-free validation pairs. Right: cross-entropy on the
tied validation pairs (dashed line: log 2, an exactly indifferent model). Mean ± sd over seeds.

In [ ]:
TP = os.path.join(ROOT, "gp_reward-priors", "calibration_results", "ties", "ties_measurements.json")
if not os.path.exists(TP):
    print("no ties measurements here yet:", TP)
else:
    T = json.load(open(TP))
    cells = [(f, v) for f in ("mr", "pt", "nmr") for v in ("large_play", "medium_diverse")]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, key, title in ((axes[0], "acc_tiefree", "accuracy, tie-free validation pairs"),
                           (axes[1], "ce_ties", "cross-entropy, tied validation pairs")):
        for k, (f, v) in enumerate(cells):
            for d, arm, mk in ((-0.15, "tiefree", "o"), (0.15, "ties", "s")):
                x = [T[f"{f}|{v}|{s}|{arm}"][key] for s in SEEDS if f"{f}|{v}|{s}|{arm}" in T]
                if x:
                    ax.errorbar(k + d, np.mean(x), yerr=np.std(x, ddof=1), fmt=mk, capsize=3,
                                color=COLOR[f.upper()], mfc="white" if arm == "tiefree" else COLOR[f.upper()])
        ax.set_xticks(range(len(cells)))
        ax.set_xticklabels([f"{f.upper()}\n{v}" for f, v in cells], fontsize=8)
        ax.set_title(title)
    axes[1].axhline(np.log(2), color="k", ls="--", lw=1)
    axes[0].legend(handles=[plt.Line2D([], [], marker="o", ls="", color="k", mfc="white", label="tie-free arm"),
                            plt.Line2D([], [], marker="s", ls="", color="k", label="ties kept")], fontsize=8)
    fig.tight_layout()

## 7. PT on the fixed labeller vs the time-only reward

Per seed, with terminals: the fixed-labeller PT cell (y) against the pre-fix cell, whose
reward depended only on time since the episode start (x; §14.10). Points on the diagonal:
the benchmark could not tell the two rewards apart.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for v in VARIANTS:
    a = D["anchor"].get((f"antmaze-{v}-v2", "PT"))
    b = D["timeonly"].get((f"antmaze-{v}-v2", "PT_TIMEONLY"))
    if ready(a) and ready(b):
        c = R.compare(a, b)
        ax.plot([b["scores"][s] for s in SEEDS], [a["scores"][s] for s in SEEDS], "o", alpha=0.8,
                label=f"{v}: fixed − time-only {c['mean']:+.3f} [{c['lower']:+.3f}, {c['upper']:+.3f}]")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlim(0, 0.9); ax.set_ylim(0, 0.9)
ax.set_xlabel("time-only reward (pre-fix PT), per seed"); ax.set_ylabel("PT, fixed labeller, per seed")
ax.legend(fontsize=7, loc="upper left")
ax.set_title("PT fixed vs time-only, with terminals")